
#  Semana 02 — Ejercicios de Optimización con PuLP

## Cuaderno de trabajo para estudiantes

Este notebook contiene **únicamente los planteamientos de los ejercicios**.  
El objetivo es que cada estudiante formule y programe su propia solución utilizando **PuLP en Python**.

---

##  Indicaciones generales

Para cada ejercicio se recomienda seguir esta secuencia:

1. Identificar las **variables de decisión**.
2. Determinar si son **continuas, enteras o binarias**.
3. Formular la **función objetivo**.
4. Escribir matemáticamente las **restricciones**.
5. Implementar el modelo en **PuLP**.
6. Resolver el modelo.
7. Revisar el estado de la solución.
8. Validar manualmente las restricciones.
9. Interpretar el resultado en el contexto del problema.

> 💡 No basta con obtener números. Debe justificarse por qué la solución encontrada es factible y qué significa en el contexto del problema.



##  Preparación del entorno

Utilice la siguiente celda únicamente para importar PuLP.

Si la librería no está instalada en su entorno, instálela antes de continuar.


In [3]:

# Importar PuLP
import pulp



#  Ejercicio 1 — Dimensionamiento de infraestructura Cloud

##  Planteamiento

Una empresa debe contratar instancias de tres tipos para soportar una nueva plataforma.  
Se desea cubrir una capacidad mínima de **CPU** y **memoria RAM** al menor costo mensual posible.

### 📊 Datos

| Tipo | Costo mensual | vCPU | RAM |
|---|---:|---:|---:|
| A — Standard | $120 | 8 | 32 GB |
| B — Compute | $180 | 16 | 64 GB |
| C — High Capacity | $260 | 32 | 96 GB |

###  Condiciones

- Se requieren al menos **160 vCPU**.
- Se requieren al menos **520 GB de RAM**.
- Por resiliencia, deben contratarse al menos **3 instancias tipo C**.
- No pueden administrarse más de **15 instancias en total**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe identificar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricciones;
- solución óptima;
- costo mínimo;
- validación de CPU, RAM y número total de instancias;
- interpretación de la solución.


In [ ]:

# EJERCICIO 1
# Escriba aquí su modelo en PuLP.

import pulp

modelo = pulp.LpProblem("Contratar_Instancias", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
x1 = pulp.LpVariable("A_Standard", lowBound=0, cat= "Integer") #lowBound (no negatividad) #cat = @GIN(lingo)
x2 = pulp.LpVariable("B_Compute", lowBound=0, cat= "Integer")
x3 = pulp.LpVariable("C_HighCapacity", lowBound=0, cat= "Integer")

#Función Objetivo
modelo += 120*x1 + 180*x2 + 260*x3,"Costo_Total"

#Restricciones
modelo += 8*x1 + 16*x2 + 32*x3 >= 160, "CPU_min"
modelo += 32*x1 + 64*x2 + 96*x3 >= 520, "Ram_min"
modelo += x3 >= 3, "Min_instancias_x3"
modelo += x1 + x2 + x3 <= 15, "Max_instancias_totales"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status]) #Estado de la optimizacion
print("Solución óptima:")
print(f"  A_x1 (Standard)     = {x1.varValue}") #valor numérico óptimo
print(f"  B_x2 (Compute)      = {x2.varValue}")
print(f"  C_x3 (HighCapacity)      = {x3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}") #resultado final de aplicar la formula de costo


Estado de la solución: Optimal
Solución óptima:
  A_x1 (Standard)     = 0.0
  B_x2 (Compute)      = 1.0
  C_x3 (HighCapacity)      = 5.0
Costo mínimo mensual = 1480.0



##  Reto de ampliación

Modifique el modelo anterior considerando ahora:

- demanda mínima de **200 vCPU**;
- demanda mínima de **640 GB de RAM**;
- obligación de contratar al menos **2 instancias tipo A** por compatibilidad con software legado.

Compare el nuevo costo con el modelo original.


In [11]:
# RETO EJERCICIO 1
import pulp

modelo = pulp.LpProblem("Contratar_Instancias", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
x1 = pulp.LpVariable("A_Standard", lowBound=0, cat= "Integer") #lowBound (no negatividad) #cat = @GIN(lingo)
x2 = pulp.LpVariable("B_Compute", lowBound=0, cat= "Integer")
x3 = pulp.LpVariable("C_HighCapacity", lowBound=0, cat= "Integer")

#Función Objetivo
modelo += 120*x1 + 180*x2 + 260*x3,"Costo_Total"

#Restricciones
modelo += 8*x1 + 16*x2 + 32*x3 >= 200, "CPU_min"
modelo += 32*x1 + 64*x2 + 96*x3 >= 640, "Ram_min"
modelo += x1 >= 2, "Min_instancias_x1"
modelo += x3 >= 3, "Min_instancias_x3"
modelo += x1 + x2 + x3 <= 15, "Max_instancias_totales"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  A_x1 (Standard)     = {x1.varValue}")
print(f"  B_x2 (Compute)      = {x2.varValue}")
print(f"  C_x3 (HighCapacity)      = {x3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  A_x1 (Standard)     = 2.0
  B_x2 (Compute)      = 0.0
  C_x3 (HighCapacity)      = 6.0
Costo mínimo mensual = 1800.0



#  Ejercicio 2 — Enrutamiento de tráfico entre enlaces WAN

##  Planteamiento

Un centro de datos debe distribuir **1,000 Mbps** entre tres enlaces WAN.  
Los enlaces tienen distintos costos, capacidades y latencias.

Se desea **minimizar el costo del tráfico**, pero la latencia promedio ponderada no debe superar **40 ms**.

###  Datos

| Enlace | Costo por Mbps | Capacidad máxima | Latencia |
|---|---:|---:|---:|
| L1 | $0.08 | 400 Mbps | 20 ms |
| L2 | $0.05 | 500 Mbps | 35 ms |
| L3 | $0.03 | 600 Mbps | 60 ms |

###  Condiciones

- Todo el tráfico debe ser enviado.
- El tráfico puede fraccionarse entre los enlaces.
- No debe superarse la capacidad máxima de cada enlace.
- La latencia promedio ponderada debe ser como máximo **40 ms**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe determinar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricción de balance;
- restricciones de capacidad;
- restricción de latencia promedio;
- costo mínimo;
- distribución óptima de tráfico;
- latencia promedio resultante.


In [ ]:
# EJERCICIO 2
# Escriba aquí su modelo en PuLP.
import pulp

modelo = pulp.LpProblem("Distribuir_mbps", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
l1 = pulp.LpVariable("Enlace1", lowBound=0, cat= "Continuous") #lowBound (no negatividad) #cat = @GIN(lingo)
l2 = pulp.LpVariable("Enlace2", lowBound=0, cat= "Continuous")
l3 = pulp.LpVariable("Enlace3", lowBound=0, cat= "Continuous")

#Función Objetivo
modelo += 0.08*l1 + 0.05*l2 + 0.03*l3,"Costo_Total"

#Restricciones
modelo += l1 + l2 +l3 == 1000, "Todo_enviarse"
modelo += l1 <= 400, "Capacidad_Max_L1"
modelo += l2 <= 500, "Capacidad_Max_L2"
modelo += l3 <= 600, "Capacidad_Max_L3"
modelo += 20*l1 + 35*l2 + 60*l3 <= 40000, "Latencia_prom_ponderada"


modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  L1 (Enlace1) = {l1.varValue}")
print(f"  L2 (Enlace2) = {l2.varValue}")
print(f"  L3 (Enlace3) = {l3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  L1 (Enlace1)     = 187.5
  L2 (Enlace2)      = 500.0
  L3 (Enlace3)      = 312.5
Costo mínimo mensual = 49.375



##  Reto de ampliación

1. Reduzca la latencia máxima permitida a **35 ms**.
2. Compare el nuevo costo con el problema original.
3. Luego simule una caída parcial de L2 reduciendo su capacidad a **200 Mbps**.
4. Determine si el problema sigue siendo factible.


In [ ]:
# RETO EJERCICIO 2
import pulp

modelo = pulp.LpProblem("Distribuir_mbps", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
l1 = pulp.LpVariable("Enlace1", lowBound=0, cat= "Continuous") #lowBound (no negatividad) #cat = continuous en lingo ya se sobreentiende (decimales)
l2 = pulp.LpVariable("Enlace2", lowBound=0, cat= "Continuous")
l3 = pulp.LpVariable("Enlace3", lowBound=0, cat= "Continuous")

#Función Objetivo
modelo += 0.08*l1 + 0.05*l2 + 0.03*l3,"Costo_Total"

#Restricciones
modelo += l1 + l2 +l3 == 1000, "Todo_enviarse"
modelo += l1 <= 400, "Capacidad_Max_L1"
modelo += l2 <= 200, "Capacidad_Max_L2"
modelo += l3 <= 600, "Capacidad_Max_L3"
modelo += 20*l1 + 35*l2 + 60*l3 <= 35000, "Latencia_prom_ponderada"


modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  L1 (Enlace1) = {l1.varValue}")
print(f"  L2 (Enlace2) = {l2.varValue}")
print(f"  L3 (Enlace3) = {l3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Infeasible
Solución óptima:
  L1 (Enlace1) = 500.0
  L2 (Enlace2) = 200.0
  L3 (Enlace3) = 300.0
Costo mínimo mensual = 59.0



#  Ejercicio 3 — Portafolio de controles de ciberseguridad

##  Planteamiento

El CISO dispone de un presupuesto limitado y debe seleccionar controles de seguridad.  
Cada control tiene un costo y una puntuación estimada de reducción de riesgo.

### 📊 Datos

| Control | Costo | Reducción de riesgo |
|---|---:|---:|
| MFA | 12 | 25 |
| EDR | 20 | 30 |
| SIEM | 25 | 28 |
| PAM | 18 | 24 |
| Backup inmutable | 15 | 22 |
| Capacitación | 8 | 12 |

###  Condiciones

- El presupuesto máximo es **70**.
- SIEM solo puede implementarse si también se selecciona EDR.
- PAM requiere que MFA esté seleccionado.
- Debe elegirse al menos una medida entre **Backup inmutable** y **Capacitación**.
- Deben seleccionarse al menos **4 controles**.

---

## Trabajo del estudiante

Construya un modelo de programación binaria que **maximice la reducción total de riesgo**.

Debe incluir:

- una variable binaria por control;
- función objetivo;
- restricción presupuestaria;
- restricciones de dependencia;
- restricción de continuidad;
- número mínimo de controles;
- interpretación de los controles seleccionados.


In [ ]:
# EJERCICIO 3
# Escriba aquí su modelo en PuLP.

import pulp

#problema
modelo = pulp.LpProblem("Selec_Control_Seguridad", pulp.LpMaximize) #maximizar reduccion de riesgo


#variables de decision
mfa = pulp.LpVariable("MFA", cat = "Binary") #Solo puede tomar 0 o 1
edr = pulp.LpVariable("EDR", cat = "Binary")
siem = pulp.LpVariable("SIEM", cat = "Binary")
pam = pulp.LpVariable("PAM", cat = "Binary")
backup = pulp.LpVariable("Backup_inmutanle", cat = "Binary")
cap = pulp.LpVariable("Capacitacion", cat = "Binary")

#funcion objetivo (riesgo)
modelo += 25*mfa + 30*edr + 28*siem + 24*pam + 22*backup + 12*cap, "Reduccion_de_riesgo_total"

#restriccion presupuestaria
modelo += 12*mfa + 20*edr + 25*siem + 18*pam + 15*backup + 8*cap <= 70, "Presupuesto_Maximo"

#restriccion de dependencia
modelo += siem <= edr, "Dependencia_SIEM_requiere_EDR"
modelo += pam <= mfa, "Dependencia_PAM_requiere_MFA"

#restriccion de al menos uno
modelo += backup + cap >= 1, "Al_menos_uno_Backup_o_Capacitacion"

#restriccion minima de eleccion
modelo += mfa + edr + siem + pam + backup + cap >= 4,"Minimo_4_controles"

modelo.solve()

print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Controles seleccionados:")
for control in [mfa, edr, siem, pam, backup, cap]:
    estado = "FUE SELECCIONADO (1)" if control.varValue == 1 else "NO FUE SELECCIONADO (0)"
    print(f"  {control.name}: {estado}")
print(f"Reducción total de riesgo = {pulp.value(modelo.objective)}")


Estado de la solución: Optimal
Controles seleccionados:
  MFA: FUE SELECCIONADO (1)
  EDR: FUE SELECCIONADO (1)
  SIEM: NO FUE SELECCIONADO (0)
  PAM: FUE SELECCIONADO (1)
  Backup_inmutanle: FUE SELECCIONADO (1)
  Capacitacion: NO FUE SELECCIONADO (0)
Reducción total de riesgo = 101.0



##  Reto de ampliación

Agregue las siguientes reglas:

1. SIEM y una herramienta *legacy* no pueden coexistir.
2. Si se elige **Backup inmutable**, también debe elegirse **MFA**.

Formule las desigualdades binarias correspondientes.

> Nota: si desea calcular un nuevo óptimo incluyendo una herramienta *legacy*, deberá definir también su costo y su contribución a la reducción de riesgo.


In [20]:
# RETO EJERCICIO 3
import pulp

#problema
modelo = pulp.LpProblem("Selec_Control_Seguridad", pulp.LpMaximize) #maximizar reduccion de riesgo


#variables de decision
mfa = pulp.LpVariable("MFA", cat = "Binary") #Solo puede tomar 0 o 1
edr = pulp.LpVariable("EDR", cat = "Binary")
siem = pulp.LpVariable("SIEM", cat = "Binary")
pam = pulp.LpVariable("PAM", cat = "Binary")
backup = pulp.LpVariable("Backup_inmutanle", cat = "Binary")
cap = pulp.LpVariable("Capacitacion", cat = "Binary")
legacy = pulp.LpVariable("Legacy", cat = "Binary")
#funcion objetivo (riesgo)
modelo += 25*mfa + 30*edr + 28*siem + 24*pam + 22*backup + 12*cap + 10*legacy, "Reduccion_de_riesgo_total"

#restriccion presupuestaria
modelo += 12*mfa + 20*edr + 25*siem + 18*pam + 15*backup + 8*cap + 5*legacy <= 70, "Presupuesto_Maximo"

#restriccion de dependencia
modelo += siem <= edr, "Dependencia_SIEM_requiere_EDR"
modelo += pam <= mfa, "Dependencia_PAM_requiere_MFA"
modelo += siem <= mfa, "Dependecia de backup_requiere_mfa"
#restriccion de al menos uno
modelo += backup + cap >= 1, "Al_menos_uno_Backup_o_Capacitacion"
#siem y legacy no coexisten
modelo += siem + legacy <= 1
#restriccion minima de eleccion
modelo += mfa + edr + siem + pam + backup + cap >= 4,"Minimo_4_controles"

modelo.solve()

print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Controles seleccionados:")
for control in [mfa, edr, siem, pam, backup, cap, legacy]:
    estado = "FUE SELECCIONADO (1)" if control.varValue == 1 else "NO FUE SELECCIONADO (0)"
    print(f"  {control.name}: {estado}")
print(f"Reducción total de riesgo = {pulp.value(modelo.objective)}")


Estado de la solución: Optimal
Controles seleccionados:
  MFA: FUE SELECCIONADO (1)
  EDR: FUE SELECCIONADO (1)
  SIEM: NO FUE SELECCIONADO (0)
  PAM: FUE SELECCIONADO (1)
  Backup_inmutanle: FUE SELECCIONADO (1)
  Capacitacion: NO FUE SELECCIONADO (0)
  Legacy: FUE SELECCIONADO (1)
Reducción total de riesgo = 111.0



#  Ejercicio 4 — Distribución de respaldos entre niveles de almacenamiento

##  Planteamiento

Una organización debe almacenar **80 TB** de respaldos utilizando tres niveles: Hot, Warm y Cold.

Se desea **minimizar el costo mensual**, manteniendo una disponibilidad mínima y un tiempo promedio de recuperación aceptable.

###  Datos

| Nivel | Costo por TB | Tiempo de recuperación |
|---|---:|---:|
| Hot | $18 | 0.5 h |
| Warm | $10 | 4 h |
| Cold | $4 | 12 h |

###  Condiciones

- El total almacenado debe ser exactamente **80 TB**.
- Al menos **15 TB** deben permanecer en Hot.
- Al menos **20 TB** deben permanecer en Warm.
- Cold no puede superar **45 TB**.
- El tiempo promedio ponderado de recuperación debe ser como máximo **8 horas**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe calcular:

- cantidad óptima de TB en cada nivel;
- costo mensual mínimo;
- tiempo promedio de recuperación;
- cumplimiento de todas las restricciones.


In [ ]:
# EJERCICIO 4
# Escriba aquí su modelo en PuLP.

import pulp

modelo = pulp.LpProblem("Distribuir_respaldos", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
n1 = pulp.LpVariable("Hot", lowBound=0, cat= "Continuous") #lowBound (no negatividad) #cat = @GIN(lingo) #continuous para decimales
n2 = pulp.LpVariable("Warm", lowBound=0, cat= "Continuous")
n3 = pulp.LpVariable("Cold", lowBound=0, cat= "Continuous")

#Función Objetivo
modelo += 18*n1 + 10*n2 + 4*n3,"Costo_Total"

#Restricciones
modelo += n1 +n2 + n3 == 80, "Total_almacenado"
modelo += n1 >= 15, "Al_menos_15TB_Hot"
modelo += n2 >= 20, "Al_menos_20TB_Warm"
modelo += n3 <= 45, "Cold_No_supera_45"
modelo += 0.5*n1 + 4*n2 + 12 * n3 <= 640, "Tiempo_promedio_ponderado"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  N1 (Nivel1) = {n1.varValue}")
print(f"  N2 (Nuvel2) = {n2.varValue}")
print(f"  N3 (Nivel3) = {n3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  N1 (Nivel1) = 15.0
  N2 (Nuvel2) = 20.0
  N3 (Nivel3) = 45.0
Costo mínimo mensual = 650.0



##  Reto de ampliación

1. Elimine la restricción que limita Cold a **45 TB**.
2. Observe si la restricción de RTO se vuelve determinante.
3. Luego exija un RTO promedio máximo de **6 horas**.
4. Compare la nueva distribución y el costo.


In [7]:
# RETO EJERCICIO 4
import pulp

modelo = pulp.LpProblem("Distribuir_respaldos", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
n1 = pulp.LpVariable("Hot", lowBound=0, cat= "Continuous") #lowBound (no negatividad) #cat = @GIN(lingo) #continuous para decimales
n2 = pulp.LpVariable("Warm", lowBound=0, cat= "Continuous")
n3 = pulp.LpVariable("Cold", lowBound=0, cat= "Continuous")

#Función Objetivo
modelo += 18*n1 + 10*n2 + 4*n3,"Costo_Total"

#Restricciones
modelo += n1 +n2 + n3 == 80, "Total_almacenado"
modelo += n1 >= 15, "Al_menos_15TB_Hot"
modelo += n2 >= 20, "Al_menos_20TB_Warm"
modelo += 0.5*n1 + 4*n2 + 12 * n3 <= 480, "Tiempo_promedio_ponderado"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  N1 (Nivel1) = {n1.varValue}")
print(f"  N2 (Nivel2) = {n2.varValue}")
print(f"  N3 (Nivel3) = {n3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  N1 (Nivel1) = 15.0
  N2 (Nivel2) = 38.4375
  N3 (Nivel3) = 26.5625
Costo mínimo mensual = 760.625



#  Ejercicio 5 — Localización de nodos Edge y asignación de regiones

##  Planteamiento

Una compañía debe decidir qué nodos Edge abrir y a qué nodo asignar cada región de usuarios.

Abrir un nodo genera un **costo fijo**.  
Atender una región desde un nodo genera un costo asociado con distancia, latencia y tráfico.

###  Nodos disponibles

| Nodo | Capacidad | Costo fijo |
|---|---:|---:|
| N1 | 80 | 100 |
| N2 | 70 | 90 |
| N3 | 75 | 95 |

### Demandas regionales

| Región | Demanda |
|---|---:|
| R1 | 40 |
| R2 | 35 |
| R3 | 30 |
| R4 | 25 |

### Costos unitarios por región y nodo

| Región | N1 | N2 | N3 |
|---|---:|---:|---:|
| R1 | 2 | 5 | 7 |
| R2 | 4 | 2 | 6 |
| R3 | 6 | 3 | 2 |
| R4 | 7 | 5 | 2 |

###  Condiciones

- Cada región debe asignarse exactamente a **un nodo**.
- Una región solo puede asignarse a un nodo que haya sido abierto.
- La suma de las demandas asignadas a cada nodo no puede superar su capacidad.
- Las decisiones de apertura y asignación son binarias.

---

##  Trabajo del estudiante

Construya un modelo que minimice:

- costos fijos de apertura;
- más costos de servicio de las regiones.

Debe determinar:

- nodos que deben abrirse;
- asignación de cada región;
- costo total;
- utilización de capacidad por nodo.


In [8]:
# EJERCICIO 5
# Escriba aquí su modelo en PuLP.

import pulp

nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

capacidad = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo = {"N1": 100, "N2": 90, "N3": 95}

demanda = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

# Costo unitario por región y nodo (matriz de costos)
costo_unitario = {
    ("R1", "N1"): 2, ("R1", "N2"): 5, ("R1", "N3"): 7,
    ("R2", "N1"): 4, ("R2", "N2"): 2, ("R2", "N3"): 6,
    ("R3", "N1"): 6, ("R3", "N2"): 3, ("R3", "N3"): 2,
    ("R4", "N1"): 7, ("R4", "N2"): 5, ("R4", "N3"): 2,
}

modelo = pulp.LpProblem("Localizacion_Nodos_Edge", pulp.LpMinimize)

# Variables de decisión

# Variables de apertura: 1 si se abre el nodo j, 0 si no
y = pulp.LpVariable.dicts("Abrir", nodos, cat="Binary")

# Variables de asignación: 1 si la región i es atendida por el nodo j
x = pulp.LpVariable.dicts("Asignar", (regiones, nodos), cat="Binary")

#Función objetivo
# Costos fijos de apertura + costos de servicio (costo unitario x demanda x asignación)

modelo += (
    pulp.lpSum(costo_fijo[j] * y[j] for j in nodos)
    + pulp.lpSum(costo_unitario[(i, j)] * demanda[i] * x[i][j] for i in regiones for j in nodos)
), "Costo_Total"

#Restricciones

# Cada región debe asignarse a exactamente un nodo
for i in regiones:
    modelo += pulp.lpSum(x[i][j] for j in nodos) == 1, f"Asignacion_unica_{i}"

# Una región solo puede asignarse a un nodo abierto
for i in regiones:
    for j in nodos:
        modelo += x[i][j] <= y[j], f"Nodo_abierto_{i}_{j}"

# La capacidad de cada nodo no puede excederse
for j in nodos:
    modelo += pulp.lpSum(demanda[i] * x[i][j] for i in regiones) <= capacidad[j], f"Capacidad_{j}"

modelo.solve()

print("Estado de la solución:", pulp.LpStatus[modelo.status])
print()

print("Nodos abiertos:")
for j in nodos:
    estado = "Abierto" if y[j].varValue == 1 else "Cerrado"
    print(f"  {j}: {estado}")

print()
print("Asignación de regiones:")
for i in regiones:
    for j in nodos:
        if x[i][j].varValue == 1:
            print(f"  {i} -> {j} (demanda: {demanda[i]}, costo unitario: {costo_unitario[(i,j)]})")

print()
print(f"Costo total mínimo = ${pulp.value(modelo.objective)}")

print()
print("Utilización de capacidad por nodo:")
for j in nodos:
    demanda_asignada = sum(demanda[i] * x[i][j].varValue for i in regiones)
    if y[j].varValue == 1:
        porcentaje = (demanda_asignada / capacidad[j]) * 100
        print(f"  {j}: {demanda_asignada}/{capacidad[j]} TB ({porcentaje:.1f}% utilizado)")
    else:
        print(f"  {j}: cerrado (0% utilizado)")

Estado de la solución: Optimal

Nodos abiertos:
  N1: Abierto
  N2: Cerrado
  N3: Abierto

Asignación de regiones:
  R1 -> N1 (demanda: 40, costo unitario: 2)
  R2 -> N1 (demanda: 35, costo unitario: 4)
  R3 -> N3 (demanda: 30, costo unitario: 2)
  R4 -> N3 (demanda: 25, costo unitario: 2)

Costo total mínimo = $525.0

Utilización de capacidad por nodo:
  N1: 75.0/80 TB (93.8% utilizado)
  N2: cerrado (0% utilizado)
  N3: 55.0/75 TB (73.3% utilizado)



##  Reto de ampliación

Analice las siguientes modificaciones:

1. Exigir que se abran al menos **2 nodos**.
2. Exigir que se abran exactamente **2 nodos**.
3. Imponer que **R1 no pueda utilizar N3** debido a un SLA de latencia.

Compare las soluciones obtenidas.


In [11]:
# RETO EJERCICIO 5
import pulp

# 2. Datos del problema

nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

capacidad = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo = {"N1": 100, "N2": 90, "N3": 95}

demanda = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

costo_unitario = {
    ("R1", "N1"): 2, ("R1", "N2"): 5, ("R1", "N3"): 7,
    ("R2", "N1"): 4, ("R2", "N2"): 2, ("R2", "N3"): 6,
    ("R3", "N1"): 6, ("R3", "N2"): 3, ("R3", "N3"): 2,
    ("R4", "N1"): 7, ("R4", "N2"): 5, ("R4", "N3"): 2,
}

# Crear el problema (minimización)
modelo = pulp.LpProblem("Localizacion_Nodos_Edge_Ampliado", pulp.LpMinimize)


# Variables de decisión

y = pulp.LpVariable.dicts("Abrir", nodos, cat="Binary")
x = pulp.LpVariable.dicts("Asignar", (regiones, nodos), cat="Binary")

# Función objetivo 

modelo += (
    pulp.lpSum(costo_fijo[j] * y[j] for j in nodos)
    + pulp.lpSum(costo_unitario[(i, j)] * demanda[i] * x[i][j] for i in regiones for j in nodos)
), "Costo_Total"

for i in regiones:
    modelo += pulp.lpSum(x[i][j] for j in nodos) == 1, f"Asignacion_unica_{i}"

for i in regiones:
    for j in nodos:
        modelo += x[i][j] <= y[j], f"Nodo_abierto_{i}_{j}"

for j in nodos:
    modelo += pulp.lpSum(demanda[i] * x[i][j] for i in regiones) <= capacidad[j], f"Capacidad_{j}"

# Modificación 1: al menos 2 nodos abiertos
modelo += pulp.lpSum(y[j] for j in nodos) >= 2, "Minimo_2_nodos"

# Modificación 2: exactamente 2 nodos abiertos

modelo += pulp.lpSum(y[j] for j in nodos) == 2, "Exactamente_2_nodos"

# Modificación 3: R1 no puede utilizar N3
modelo += x["R1"]["N3"] == 0, "R1_no_usa_N3"

modelo.solve()

print("Estado de la solución:", pulp.LpStatus[modelo.status])
print()

print("Nodos abiertos:")
for j in nodos:
    estado = "Abierto" if y[j].varValue == 1 else "Cerrado"
    print(f"  {j}: {estado}")

print()
print("Asignación de regiones:")
for i in regiones:
    for j in nodos:
        if x[i][j].varValue == 1:
            print(f"  {i} -> {j} (demanda: {demanda[i]}, costo unitario: {costo_unitario[(i,j)]})")

print()
print(f"Costo total mínimo = ${pulp.value(modelo.objective)}")

Estado de la solución: Optimal

Nodos abiertos:
  N1: Abierto
  N2: Cerrado
  N3: Abierto

Asignación de regiones:
  R1 -> N1 (demanda: 40, costo unitario: 2)
  R2 -> N1 (demanda: 35, costo unitario: 4)
  R3 -> N3 (demanda: 30, costo unitario: 2)
  R4 -> N3 (demanda: 25, costo unitario: 2)

Costo total mínimo = $525.0



#  Ejercicio 6 — Dimensionamiento de agentes de CI/CD

## Planteamiento

Una plataforma DevOps necesita capacidad concurrente para pipelines Linux y Windows.

Existen tres tipos de agentes con diferentes capacidades y costos.

###  Datos

| Tipo de agente | Linux slots | Windows slots | Costo |
|---|---:|---:|---:|
| Standard | 4 | 2 | 50 |
| Linux Optimized | 8 | 0 | 70 |
| Universal | 3 | 5 | 80 |

### Condiciones

- Se requieren al menos **40 slots Linux**.
- Se requieren al menos **20 slots Windows**.
- Deben existir al menos **2 agentes Universal**.
- El equipo de operaciones puede administrar como máximo **12 agentes**.

---

##  Trabajo del estudiante

Formule y resuelva un modelo de programación entera que minimice el costo total.

Debe determinar:

- cantidad de agentes Standard;
- cantidad de agentes Linux Optimized;
- cantidad de agentes Universal;
- costo mínimo;
- slots Linux obtenidos;
- slots Windows obtenidos;
- total de agentes utilizados.


In [12]:
# EJERCICIO 6
# Escriba aquí su modelo en PuLP.

import pulp

modelo = pulp.LpProblem("Seleccion_Agentes", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
x1 = pulp.LpVariable("Standard", lowBound=0, cat= "Integer") #lowBound (no negatividad) #cat = @GIN(lingo) Integer (entero)
x2 = pulp.LpVariable("Linux_Optimized", lowBound=0, cat= "Integer")
x3 = pulp.LpVariable("Universal", lowBound=0, cat= "Integer")

#Función Objetivo
modelo += 50*x1 + 70*x2 + 80*x3,"Costo_Total"

#Restricciones
modelo += 4*x1 + 8*x2 + 3*x3 >= 40, "Slots_Linux"
modelo += 2*x1 + 0*x2 + 5*x3 >= 20, "Slots_Windows"
modelo += x3 >= 2, "Min_AG_Universales"
modelo += x1 + x2 + x3 <= 12, "Max_AG_Totales"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  X1 (Standard) = {x1.varValue}")
print(f"  X2 (Linux_Optimized) = {x2.varValue}")
print(f"  X3 (Universal) = {x3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  X1 (Standard) = 5.0
  X2 (Linux_Optimized) = 2.0
  X3 (Universal) = 2.0
Costo mínimo mensual = 550.0



## Reto de ampliación

Modifique el modelo de la siguiente manera:

1. Aumente el requerimiento de Windows a **30 slots**.
2. Agregue la regla:

> Por cada 3 agentes Linux Optimized debe existir al menos 1 agente Universal.

Formule matemáticamente dicha restricción e incorpórela al modelo.

Compare el nuevo costo con el problema original.


In [13]:
# RETO EJERCICIO 6

import pulp

modelo = pulp.LpProblem("Seleccion_Agentes", pulp.LpMinimize) #Minimizacion de costos (problema)

# Nuestras Variables
x1 = pulp.LpVariable("Standard", lowBound=0, cat= "Integer") #lowBound (no negatividad) #cat = @GIN(lingo) Integer (entero)
x2 = pulp.LpVariable("Linux_Optimized", lowBound=0, cat= "Integer")
x3 = pulp.LpVariable("Universal", lowBound=0, cat= "Integer")

#Función Objetivo
modelo += 50*x1 + 70*x2 + 80*x3,"Costo_Total"

#Restricciones
modelo += 4*x1 + 8*x2 + 3*x3 >= 40, "Slots_Linux"
modelo += 2*x1 + 0*x2 + 5*x3 >= 30, "Slots_Windows"
modelo += x3 >= 2, "Min_AG_Universales"
modelo += x1 + x2 + x3 <= 12, "Max_AG_Totales"
modelo += x2 <= 3*x3, "Por_cada_3_linux_1_universal"

modelo.solve()

#RESULTADOS
print("Estado de la solución:", pulp.LpStatus[modelo.status])
print("Solución óptima:")
print(f"  X1 (Standard) = {x1.varValue}")
print(f"  X2 (Linux_Optimized) = {x2.varValue}")
print(f"  X3 (Universal) = {x3.varValue}")
print(f"Costo mínimo mensual = {pulp.value(modelo.objective)}")

Estado de la solución: Optimal
Solución óptima:
  X1 (Standard) = 8.0
  X2 (Linux_Optimized) = 0.0
  X3 (Universal) = 3.0
Costo mínimo mensual = 640.0



#  Entrega sugerida

Para cada ejercicio, entregue:

- formulación matemática;
- código en PuLP;
- estado del solver;
- valores de las variables;
- valor de la función objetivo;
- comprobación de restricciones;
- interpretación breve de la solución.

> Si el estado del modelo no es `Optimal`, no interprete los valores de las variables como una solución óptima.
